In [88]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [89]:
import os
from runner import DualRunner
import pandas as pd

PG_CONNINFO = (
    f"host=127.0.0.1 "
    f"port={os.getenv("POSTGRES_PORT", 5432)} "
    f"dbname={os.getenv("POSTGRES_DB")} "
    f"user={os.getenv("POSTGRES_USER")} "
    f"password={os.getenv("POSTGRES_PASSWORD")}"
)

runner = DualRunner(
    pg_conninfo=PG_CONNINFO,
    duckdb_path=":memory:"
)

display(runner.run_pg("select version()"))
display(runner.run_dd("select version()"))

## pandas 設定

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

,version
0,"PostgreSQL 17.7 (Debian 17.7-3.pgdg13+1) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit"


,"""version""()"
0,v1.4.3


# データ加工のためのSQL
## 一つの値に対する処理

In [90]:
runner.check("""--sql
drop table if exists access_log;
create table access_log (
    stamp timestamp,
    referrer text,
    url text
);
insert into access_log (stamp, referrer, url) values
('2016-08-26 12:02:00', 'http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1', 'http://www.example.com/video/detail?id=001'),
('2016-08-26 12:02:01', 'http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1', 'http://www.example.com/video#ref'),
('2016-08-26 12:02:01', 'https://www.other.com/', 'http://www.example.com/book/detail?id=002');            
select * from access_log;
             
""")

### ✅ SAME

,stamp,referrer,url
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002


In [91]:
runner.pg("""--sql

-- 正規表現を使って値を抽出する
select
    stamp,
    referrer,
    url,
    substring(referrer from 'https?://([^/]*)') as referrer_domain,
    substring(url from '//[^/]+([^?#]+)') as path,
    substring(url from 'id=([^&]*)') as id
from access_log
""")

runner.dd("""--sql

select
    stamp,
    referrer,
    url,
    regexp_extract(referrer, 'https?://([^/]*)', 1) as referrer_domain,
    regexp_extract(url,  '//[^/]+([^?#]+)', 1) as path,
    regexp_extract(url,  'id=([^&]*)', 1) as id
from access_log

""")

### 🐘 PostgreSQL Result

,stamp,referrer,url,referrer_domain,path,id
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001,www.other.com,/video/detail,001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref,www.other.net,/video,None
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002,www.other.com,/book/detail,002


### 🦆 DuckDB Result

,stamp,referrer,url,referrer_domain,path,id
0,2016-08-26 12:02:00,http://www.other.com/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video/detail?id=001,www.other.com,/video/detail,001
1,2016-08-26 12:02:01,http://www.other.net/path1/index.php?k1=v1&k2=v2#Ref1,http://www.example.com/video#ref,www.other.net,/video,
2,2016-08-26 12:02:01,https://www.other.com/,http://www.example.com/book/detail?id=002,www.other.com,/book/detail,002


## 文字列を配列に分解する

`split_part(str, '/', 2)` などと書けば、分割する文字と分割した後にインデックスを指定して抽出できる。duckdb でも同じ関数。




In [92]:
runner.pg("""--sql
-- ulr のパスをスラッシュで分割して階層を抽出する
select
    stamp,
    url,
    split_part(substring(url from '//[^/]+([^?#]+)'), '/', 2) as path_1,
    split_part(substring(url from '//[^/]+([^?#]+)'), '/', 3) as path_2
from access_log
""")

runner.dd("""--sql
-- ulr のパスをスラッシュで分割して階層を抽出する
select
    stamp,
    url,
    split_part(regexp_extract(url, '//[^/]+([^?#]+)', 1), '/', 2) as path_1,
    split_part(regexp_extract(url, '//[^/]+([^?#]+)', 1), '/', 3) as path_2
from access_log
""")

### 🐘 PostgreSQL Result

,stamp,url,path_1,path_2
0,2016-08-26 12:02:00,http://www.example.com/video/detail?id=001,video,detail
1,2016-08-26 12:02:01,http://www.example.com/video#ref,video,
2,2016-08-26 12:02:01,http://www.example.com/book/detail?id=002,book,detail


### 🦆 DuckDB Result

,stamp,url,path_1,path_2
0,2016-08-26 12:02:00,http://www.example.com/video/detail?id=001,video,detail
1,2016-08-26 12:02:01,http://www.example.com/video#ref,video,
2,2016-08-26 12:02:01,http://www.example.com/book/detail?id=002,book,detail


## 日付やタイムスタンプを扱う


In [93]:
runner.check("""--sql
select
    current_date as today,

    -- これはTZ付きデータになる
    current_timestamp as now_with_tz,

    -- localtimestamp や TZを指定すると、TZ無しデータになる
    localtimestamp as local_now,
    current_timestamp at time zone 'UTC' as now_utc,
    current_timestamp at time zone 'Asia/Tokyo' as now_tokyo,

    -- current_timestamp の代わりに now() を使っても同じ
    now() as now_with_tz_2,
    now() at time zone 'UTC' as now_utc_2,
    now() at time zone 'Asia/Tokyo' as now_tokyo_2
""")

### ✅ SAME

,today,now_with_tz,local_now,now_utc,now_tokyo,now_with_tz_2,now_utc_2,now_tokyo_2
0,2026-01-21,2026-01-21 20:33:21.233777+09:00,2026-01-21 20:33:21.233777,2026-01-21 11:33:21.233777,2026-01-21 20:33:21.233777,2026-01-21 20:33:21.233777+09:00,2026-01-21 11:33:21.233777,2026-01-21 20:33:21.233777


In [94]:
runner.check("""--sql
select
    '2016-08-26 12:02:00+09'::timestamptz as ts_with_tz,
    '2016-08-26 12:02:00'::timestamp as ts_without_tz,
    '2016-08-26'::date as only_date,

    -- cast で変換することも可能
    cast('2016-08-26 12:02:00+09' as timestamptz) as ts_with_tz_cast,
    cast('2016-08-26 12:02:00' as timestamp) as ts_without_tz_cast,
    cast('2016-08-26' as date) as only_date_cast,

    -- 以下の書き方もできる
    timestamptz '2016-08-26 12:02:00+09' as ts_with_tz_literal,
    timestamp '2016-08-26 12:02:00' as ts_without_tz_literal,
    date '2016-08-26' as only_date_literal
""")

### ✅ SAME

,ts_with_tz,ts_without_tz,only_date,ts_with_tz_cast,ts_without_tz_cast,only_date_cast,ts_with_tz_literal,ts_without_tz_literal,only_date_literal
0,2016-08-26 12:02:00+09:00,2016-08-26 12:02:00,2016-08-26,2016-08-26 12:02:00+09:00,2016-08-26 12:02:00,2016-08-26,2016-08-26 12:02:00+09:00,2016-08-26 12:02:00,2016-08-26


### 日付・時刻から特定のフィールドを取り出す


In [95]:
runner.check("""--sql
             
with t as (
    select 
        '2016-08-26 12:02:39'::timestamp as stamp,
        '2016-08-26 12:02:39.82943'::timestamp as stamp_ms
        
)

select
    stamp,
    stamp_ms,
    extract(year from stamp) as year,
    extract(month from stamp) as month,
    extract(day from stamp) as day,
    extract(hour from stamp) as hour,
    extract(minute from stamp) as minute,

    -- second のあつかいが PG と duckdb で異なる
    -- second は duckb では小数点以下を切り捨てるが、PG では小数点以下も含む
    extract(second from stamp) as second, 
    extract(second from stamp_ms) as second_ms,

    extract(microsecond from stamp_ms) as microsecond 
from t
             
             
""")

## ❌ DIFF Detected

#### 🐘 PostgreSQL Result

,stamp,stamp_ms,year,month,day,hour,minute,second,second_ms,microsecond
0,2016-08-26 12:02:39,2016-08-26 12:02:39.829430,2016,8,26,12,2,39.000000,39.829430,39829430


#### 🦆 DuckDB Result

,stamp,stamp_ms,year,month,day,hour,minute,second,second_ms,microsecond
0,2016-08-26 12:02:39,2016-08-26 12:02:39.829430,2016,8,26,12,2,39,39,39829430


PG dtypes:
stamp          datetime64[ns]
stamp_ms       datetime64[ns]
year                   object
month                  object
day                    object
hour                   object
minute                 object
second                 object
second_ms              object
microsecond            object
dtype: object

DuckDB dtypes:
stamp          datetime64[us]
stamp_ms       datetime64[us]
year                    int64
month                   int64
day                     int64
hour                    int64
minute                  int64
second                  int64
second_ms               int64
microsecond             int64
dtype: object


### 欠損値をデフォルト値に置き換える

In [96]:
runner.check("""--sql 
drop table if exists purchase_log_with_coupon;
create table purchase_log_with_coupon (
    purchase_id integer,
    amount integer,
    coupon integer
);
insert into purchase_log_with_coupon (purchase_id, amount, coupon) values
(10001, 3280, null),
(10002, 4650, 500),
(10003, 3870, null);
select * from purchase_log_with_coupon;
             
""")

### ✅ SAME

,purchase_id,amount,coupon
0,10001,3280,NaN
1,10002,4650,500.0
2,10003,3870,NaN


In [97]:
#* 購入額から割引クーポンを引いて、実際の支払額を計算する
runner.check("""--sql
select
    purchase_id,
    amount,
    coupon,
    amount - coupon as actual_payment_pg, --null を四則演算すると null になるので注意
    amount - coalesce(coupon, 0) as actual_payment -- colalesce で null を 0 に変換してから計算する
from purchase_log_with_coupon
""")

### ✅ SAME

,purchase_id,amount,coupon,actual_payment_pg,actual_payment
0,10001,3280,NaN,NaN,3280
1,10002,4650,500.0,4150.0,4150
2,10003,3870,NaN,NaN,3870


## 複数の値に対する操作

### 文字列の連結

In [98]:
runner.check("""--sql

drop table if exists mst_user_location;
create table mst_user_location (
    user_id varchar(10),
    pref_name varchar(50),
    city_name varchar(50)
);
insert into mst_user_location (user_id, pref_name, city_name) values
('U001', '東京都', '千代田区'),
('U002', '東京都', '渋谷区'),
('U003', '千葉県', '八千代市');
select * from mst_user_location;

""")

### ✅ SAME

,user_id,pref_name,city_name
0,U001,東京都,千代田区
1,U002,東京都,渋谷区
2,U003,千葉県,八千代市


In [ ]:
runner.check("""--sql
             
-- 都道府県と市区町村を結合してフル住所を作成する
-- concat もしくは || 演算子を使う
select
    concat(pref_name, city_name) as full_address,
    pref_name || city_name as full_address_2
from mst_user_location
""")

### ✅ SAME

,full_address,full_address_2
0,東京都千代田区,東京都千代田区
1,東京都渋谷区,東京都渋谷区
2,千葉県八千代市,千葉県八千代市


In [101]:
runner.check("""--sql
             
drop table if exists quarterly_sales;
create table quarterly_sales (
    year integer,
    q1 integer,
    q2 integer,
    q3 integer,
    q4 integer
);
insert into quarterly_sales (year, q1, q2, q3, q4) values
(2015, 82000, 83000, 78000, 83000),
(2016, 85000, 85000, 80000, 81000),
(2017, 92000, 81000, null, null);
select * from quarterly_sales;
""")

### ✅ SAME

,year,q1,q2,q3,q4
0,2015,82000,83000,78000.0,83000.0
1,2016,85000,85000,80000.0,81000.0
2,2017,92000,81000,NaN,NaN


In [ ]:
runner.check("""--sql

select
    year,
    q1, q2, q3, q4,

    -- 場合分け
    case
        when q1 < q2 then '+'
        when q1 = q2 then ' '
        when q1 > q2 then '-'
    end as judge,

    q2 - q1 as diff,

    -- sign(x)は x が正なら 1、負なら -1、0なら 0 を返す 
    sign(q2 - q1) as sign,

    -- 複数カラムの値の最大最小
    greatest(q1, q2, q3, q4) as greatest,
    least(q1, q2, q3, q4) as least,
             
    -- 平均は専用の関数はないので手動で数式をつくって　計算
    -- DB によって表示桁数の違いがあるので注意
    -- null があると四則演算しても null
    (q1 + q2 + q3 + q4) / 4.0 as avg,

    -- nullを除いて 平均を計算するには
    -- 分子は coalesce で null を 0 に変換 しながら、
    --分母は null -> 0 に変換した上で sign() で 0 or 1 に変換して合計する
    (coalesce(q1,0) + coalesce(q2,0) + coalesce(q3,0) + coalesce(q4,0)) /
    (
        sign(coalesce(q1,0)) + sign(coalesce(q2,0)) +
        sign(coalesce(q3,0)) + sign(coalesce(q4,0))
    ) as avg_ignore_null

from quarterly_sales
order by year
""")

## ❌ DIFF Detected

#### 🐘 PostgreSQL Result

,year,q1,q2,q3,q4,judge,diff,sign,greatest,least,avg,avg_ignore_null
0,2015,82000,83000,78000.0,83000.0,+,1000,1.0,83000,78000,81500.000000000000,81500.0
1,2016,85000,85000,80000.0,81000.0,,0,0.0,85000,80000,82750.000000000000,82750.0
2,2017,92000,81000,NaN,NaN,-,-11000,-1.0,92000,81000,None,86500.0


#### 🦆 DuckDB Result

,year,q1,q2,q3,q4,judge,diff,sign,greatest,least,avg,avg_ignore_null
0,2015,82000,83000,78000,83000,+,1000,1,83000,78000,81500.0,81500.0
1,2016,85000,85000,80000,81000,,0,0,85000,80000,82750.0,82750.0
2,2017,92000,81000,<NA>,<NA>,-,-11000,-1,92000,81000,NaN,86500.0


PG dtypes:
year                 int64
q1                   int64
q2                   int64
q3                 float64
q4                 float64
judge               object
diff                 int64
sign               float64
greatest             int64
least                int64
avg                 object
avg_ignore_null    float64
dtype: object

DuckDB dtypes:
year                 int32
q1                   int32
q2                   int32
q3                   Int32
q4                   Int32
judge               object
diff                 int32
sign                  int8
greatest             int32
least                int32
avg                float64
avg_ignore_null    float64
dtype: object


**一般に複数のカラムを使った計算は面倒になるので、縦持ちに変換した後に集計したほうが良い**

## 2つの値の比率を計算する

In [123]:
runner.check("""--sql 

drop table if exists advertising_stats;
create table advertising_stats (
    dt date,
    ad_id varchar(10),
    impressions integer,
    clicks integer
);
insert into advertising_stats (dt, ad_id, impressions, clicks) values
('2017-04-01', '001', 100000, 3000),
('2017-04-01', '002', 120000, 1200),
('2017-04-01', '003', 500000, 10000),
('2017-04-02', '001', 0, 0),
('2017-04-02', '002', 130000, 1400),
('2017-04-02', '003', 620000, 15000);
select * from advertising_stats;
""")

### ✅ SAME

,dt,ad_id,impressions,clicks
0,2017-04-01,001,100000,3000
1,2017-04-01,002,120000,1200
2,2017-04-01,003,500000,10000
3,2017-04-02,001,0,0
4,2017-04-02,002,130000,1400
5,2017-04-02,003,620000,15000


In [ ]:
runner.check("""--sql 
             
-- click through rate (CTR) を計算する
select
    dt,
    ad_id,
    -- clicks / impressions as ctr -- データに0が含まれるのでエラーになる
    -- そのため nullif(x, 0) を使って 0 の場合は null に変換してから計算する
    clicks::numeric / nullif(impressions, 0)::numeric as ctr_numeric,
    clicks::double precision / nullif(impressions, 0)::double precision  as ctr_double, 
    clicks::decimal / nullif(impressions, 0)::decimal  as ctr_decimal,
    (clicks*1.0) / (nullif(impressions, 0)*1.0)  as ctr_1,
    clicks::float / nullif(impressions, 0)::float  as ctr_float  -- これが両方で一番使いやすいか


from advertising_stats




""")

## ❌ DIFF Detected

#### 🐘 PostgreSQL Result

,dt,ad_id,ctr_numeric,ctr_double,ctr_decimal,ctr_1,ctr_float
0,2017-04-01,001,0.03000000000000000000,0.030000,0.03000000000000000000,0.03000000000000000000,0.030000
1,2017-04-01,002,0.01000000000000000000,0.010000,0.01000000000000000000,0.01000000000000000000,0.010000
2,2017-04-01,003,0.02000000000000000000,0.020000,0.02000000000000000000,0.02000000000000000000,0.020000
3,2017-04-02,001,None,NaN,None,None,NaN
4,2017-04-02,002,0.01076923076923076923,0.010769,0.01076923076923076923,0.01076923076923076923,0.010769
5,2017-04-02,003,0.02419354838709677419,0.024194,0.02419354838709677419,0.02419354838709677419,0.024194


#### 🦆 DuckDB Result

,dt,ad_id,ctr_numeric,ctr_double,ctr_decimal,ctr_1,ctr_float
0,2017-04-01,001,0.030000,0.030000,0.030000,0.030000,0.030000
1,2017-04-01,002,0.010000,0.010000,0.010000,0.010000,0.010000
2,2017-04-01,003,0.020000,0.020000,0.020000,0.020000,0.020000
3,2017-04-02,001,NaN,NaN,NaN,NaN,NaN
4,2017-04-02,002,0.010769,0.010769,0.010769,0.010769,0.010769
5,2017-04-02,003,0.024194,0.024194,0.024194,0.024194,0.024194


PG dtypes:
dt              object
ad_id           object
ctr_numeric     object
ctr_double     float64
ctr_decimal     object
ctr_1           object
ctr_float      float64
dtype: object

DuckDB dtypes:
dt             datetime64[us]
ad_id                  object
ctr_numeric           float64
ctr_double            float64
ctr_decimal           float64
ctr_1                 float64
ctr_float             float32
dtype: object


## 2つの値の距離を計算する

2つの値の距離を計算する方法は2つ
- 絶対値
- 二乗平均平方根 (RMS)
    - 差の二乗の平方根

In [ ]:
runner.check("""--sql
drop table if exists location_1d;
create table location_1d (
    x1 integer,
    x2 integer
);
insert into location_1d (x1, x2) values
    (5, 10),
    (10, 5),
    (-2, 4),
    (3, 3),
    (0, 1);
select * from location_1d;
""")

### ✅ SAME

,x1,x2
0,5,10
1,10,5
2,-2,4
3,3,3
4,0,1


In [ ]:
runner.check("""--sql
             
-- 1次元上の2点間の距離を計算する
select
    x1, x2,
    abs(x1 - x2) as abs_diff,
    sqrt(power(x1 - x2, 2)) as rms_diff
from location_1d
""")

### ✅ SAME

,x1,x2,abs_diff,rms_diff
0,5,10,5,5.0
1,10,5,5,5.0
2,-2,4,6,6.0
3,3,3,0,0.0
4,0,1,1,1.0


In [146]:
runner.check("""--sql
drop table if exists location_2d;
create table location_2d (
    x1 integer,
    y1 integer,
    x2 integer,
    y2 integer
);
insert into location_2d (x1, y1, x2, y2) values
(0, 0, 2, 2),
(3, 5, 1, 2),
(5, 3, 2, 1);            
select * from location_2d;
""")

### ✅ SAME

,x1,y1,x2,y2
0,0,0,2,2
1,3,5,1,2
2,5,3,2,1


In [149]:
runner.check("""--sql
             
select
    sqrt(power(x1 - x2, 2) + power(y1 - y2, 2)) as distance_2d
    --point(x1, y1) <-> point(x2, y2) as distance_2d_pg -- PostgreSQL 専用関数
from location_2d
""")

### ✅ SAME

,distance_2d
0,2.828427
1,3.605551
2,3.605551


## 日付・時刻を計算する

In [150]:
runner.check("""--sql
drop table if exists mst_users_with_dates;
create table mst_users_with_dates (
    user_id varchar(10),
    register_stamp timestamp,
    birth_date date
);
insert into mst_users_with_dates (user_id, register_stamp, birth_date) values
('U001', '2016-02-28 10:00:00', '2000-02-29'),
('U002', '2016-02-29 10:00:00', '2000-02-29'),
('U003', '2016-03-01 10:00:00', '2000-02-29');
select * from mst_users_with_dates;
""")

### ✅ SAME

,user_id,register_stamp,birth_date
0,U001,2016-02-28 10:00:00,2000-02-29
1,U002,2016-02-29 10:00:00,2000-02-29
2,U003,2016-03-01 10:00:00,2000-02-29


In [ ]:
runner.check("""--sql

-- 足し引き
select
    user_id,
    -- interval 型を使って日時の加減算を行う
    register_stamp::timestamp as timestamp,
    register_stamp::timestamp + '1 hour'::interval as stamp_after_1hour,
    register_stamp::timestamp - '30 minutes'::interval as stamp_before_30min,
    
    birth_date::date as birth_date,
    birth_date::date + '1 day'::interval as birth_date_after_1year,
    birth_date::date - '1 month'::interval as birth_date_before_1month
             
from mst_users_with_dates
             

""")

### ✅ SAME

,user_id,timestamp,stamp_after_1hour,stamp_before_30min,birth_date,birth_date_after_1year,birth_date_before_1month,today,register_date,days_since_register
0,U001,2016-02-28 10:00:00,2016-02-28 11:00:00,2016-02-28 09:30:00,2000-02-29,2000-03-01,2000-01-29,2026-01-22,2016-02-28,3616
1,U002,2016-02-29 10:00:00,2016-02-29 11:00:00,2016-02-29 09:30:00,2000-02-29,2000-03-01,2000-01-29,2026-01-22,2016-02-29,3615
2,U003,2016-03-01 10:00:00,2016-03-01 11:00:00,2016-03-01 09:30:00,2000-02-29,2000-03-01,2000-01-29,2026-01-22,2016-03-01,3614


In [ ]:
runner.check("""--sql

select
    user_id,
    -- データ同士の差分
    -- 両方 date 型に合わせてから引き算する
    current_date as today,
    register_stamp::date as register_date,
    current_date - register_stamp::date as days_since_register
from mst_users_with_dates
             

""")

### ✅ SAME

,user_id,today,register_date,days_since_register
0,U001,2026-01-22,2016-02-28,3616
1,U002,2026-01-22,2016-02-29,3615
2,U003,2026-01-22,2016-03-01,3614


In [167]:
runner.check("""--sql
             

select
    user_id,
    current_date as today,
    register_stamp::date as register_date,
    birth_date::date as birth_date,
    '2025-06-15'::date as some_date,
             
    -- 年齢を計算する
    -- 両者に age() 関数が存在する
    -- age() 関数は2つの日付の差分を interval 型で返す
    -- 本日との差分を計算する場合は current_date を省略できる
    age(current_date, birth_date::date) as age_interval,
    date_part('year', age(birth_date::date)) as current_age_short,
    -- これと同じ
    date_part('year', age(current_date, birth_date::date)) as current_age,

    date_part('year', age(register_stamp::date, birth_date::date)) as age_at_register
from mst_users_with_dates


""")

## ❌ DIFF Detected

#### 🐘 PostgreSQL Result

,user_id,today,register_date,birth_date,some_date,age_interval,current_age_short,current_age,age_at_register
0,U001,2026-01-22,2016-02-28,2000-02-29,2025-06-15,9447 days,25.0,25.0,15.0
1,U002,2026-01-22,2016-02-29,2000-02-29,2025-06-15,9447 days,25.0,25.0,16.0
2,U003,2026-01-22,2016-03-01,2000-02-29,2025-06-15,9447 days,25.0,25.0,16.0


#### 🦆 DuckDB Result

,user_id,today,register_date,birth_date,some_date,age_interval,current_age_short,current_age,age_at_register
0,U001,2026-01-22,2016-02-28,2000-02-29,2025-06-15,9322 days,25,25,15
1,U002,2026-01-22,2016-02-29,2000-02-29,2025-06-15,9322 days,25,25,16
2,U003,2026-01-22,2016-03-01,2000-02-29,2025-06-15,9322 days,25,25,16


PG dtypes:
user_id                       object
today                         object
register_date                 object
birth_date                    object
some_date                     object
age_interval         timedelta64[ns]
current_age_short            float64
current_age                  float64
age_at_register              float64
dtype: object

DuckDB dtypes:
user_id                       object
today                 datetime64[us]
register_date         datetime64[us]
birth_date            datetime64[us]
some_date             datetime64[us]
age_interval         timedelta64[ns]
current_age_short              int64
current_age                    int64
age_at_register                int64
dtype: object
